# 2) Herd-specific technological trends
- This code creates a variable "adj_year" for data split later, ensuring complete lactation sequences (Supplementary Information).
- It plots county-level cow population and milk averages (Supplementary Information).
- It finally removes herd-specific technolgoical trends from milk.

# 1. Import pacakges

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import seaborn as sns
import glob
import os, sys, gc
from pathlib import Path

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
pd.set_option('display.max_columns', None)

# 2. Reading and Cleaning data:

## 2-1. Opening milk yield data:

In [ ]:
## reading/cleaning:
df = pd.read_parquet('1_data/final_synthetic_df.gzip').rename(columns={'herd_code':'herd','cow_id':'id','calving_date':'calvingdate',
                                                                       'parity':'lac_num','days_in_milk':'dim','cal_yr':'year','milk':'adj_milk_kg'}).drop(columns=['year_month'])

df.columns

## 2-2. Formatting and Cleaning:
- At the time of downloading the weather data, PRISM data was only available until June. Limit our analysis to Jan 2000 to Jun 2024.
- In DRMS data, milk is reported in pounds, converting to kg and to energy corrected milk ("adj_milk_kg").

### a. Formatting:

In [ ]:
df['geoid_herd'] = df['GEOID'] + df['herd']
df = df.loc[df['date'] <="2024-06-30"].reset_index(drop=True)
df['cal_yr'] = df['calvingdate'].dt.year

In [ ]:
## choose cows having more than 3 dims based on the minimum number of representing a full lac cycle. We already filter this step, so check 2024 (calving year >= 2023) only:
print(df.shape[0])
df.loc[df['cal_yr'] >= 2023, 'dim_count'] =df.loc[df['cal_yr'] >= 2023].groupby(['GEOID','geoid_herd','id','lac_num'])['dim'].transform('count')
df = df.loc[~((df['cal_yr'] >= 2023) & (df['dim_count'] <3))].reset_index(drop=True)
df.shape[0]

### b. Creating adj_year variable (Supplementary Information)
- In order to prevent data leakage of training data into validation or test, this variable is required.
- Main assumption is that cows having more test records in calving years are fully under calving year's management;
- cows calving before Sept have the calving year management impacts.

In [ ]:
## creating lac_year:
df['dim_count'] = df.groupby(['state_abv','GEOID','geoid_herd','id','calvingdate'])['dim'].transform('nunique')
lac_stats = df[['GEOID','geoid_herd','id','calvingdate','cal_yr','year','dim','dim_count']].drop_duplicates()
## count number of years that are same as calving year
lac_stats.loc[lac_stats['cal_yr'] == lac_stats['year'], 'cal_yr_count'] = lac_stats.loc[lac_stats['cal_yr'] == lac_stats['year']].groupby(['geoid_herd','id','calvingdate'])['year'].transform('count')
lac_stats = lac_stats[['GEOID','geoid_herd','id','calvingdate','dim_count','cal_yr_count']].drop_duplicates(subset=['geoid_herd','id','calvingdate','dim_count'])
## ratio shows the number of years that are same as calving year
lac_stats['ratio'] = lac_stats['cal_yr_count'] / lac_stats['dim_count']
lac_stats.loc[lac_stats['ratio'] != 1]

In [ ]:
## ca_yr => calving year 
lac_stats['cal_yr'] = lac_stats['calvingdate'].dt.year

### all test-day records within calving_year:
lac_stats['adj_year'] = np.nan
# If most or all of the lactation occurred in the calving year, lac_yr = cal_r:
lac_stats.loc[(lac_stats['ratio'] ==1) | (lac_stats['ratio'] >= 0.5), 'adj_year'] =  lac_stats.loc[(lac_stats['ratio'] ==1) | (lac_stats['ratio'] >= 0.5)]['cal_yr']

# If ~40–50% of test-day records are in the calving year and the cow calved early in the year (before September), lac_yr = cal_yr:
''' For cows having a slightly less number of recrds in calving year but calved before Oct, they are influenced by feed or management plans of that calving year'''
lac_stats.loc[(lac_stats['ratio'] >= 0.4) & (lac_stats['ratio'] < 0.5) & (lac_stats['calvingdate'].dt.month < 9),'adj_year'] = lac_stats.loc[(lac_stats['ratio'] >= 0.4) & (lac_stats['ratio'] < 0.5) & (lac_stats['calvingdate'].dt.month < 9)]['cal_yr']

### consider these as next year management (50% of test records fall in the next year)
# Otherwise (few or no records in the calving year), assign to next year
lac_stats.loc[(lac_stats['ratio'].isnull()) | (lac_stats['ratio'] < 0.4), 'adj_year'] =  lac_stats.loc[(lac_stats['ratio'].isnull()) | (lac_stats['ratio'] < 0.4)]['cal_yr'] +1
# For late calving cows (~Sept–Dec) with 40–50% records in calving year, also assign to next year:
lac_stats.loc[(lac_stats['ratio'] >= 0.4) & (lac_stats['ratio'] < 0.5) & (lac_stats['calvingdate'].dt.month >= 9), 'adj_year'] = lac_stats.loc[(lac_stats['ratio'] >= 0.4) & (lac_stats['ratio'] < 0.5) & (lac_stats['calvingdate'].dt.month >= 9)]['cal_yr']+1

lac_stats.loc[lac_stats['adj_year'].isnull()]

In [ ]:
## checking lac_stats capture all:
lac_stats.shape[0], df[['geoid_herd','id','lac_num']].drop_duplicates().shape[0]

In [ ]:
## writing back to df
print(df.shape[0])
df = df.merge(lac_stats[['GEOID','geoid_herd','id','calvingdate','adj_year']].drop_duplicates(),
                                                                on=['GEOID','geoid_herd','id','calvingdate'], how='left')
df.shape[0]

In [ ]:
## cleaning
del lac_stats
gc.collect()

## 2-3. Spatial map of milk yields and cow population (Supplementary Infomation):


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
with urlopen('https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json') as response:
    states = json.load(response)

# remove HI, AK, PR
states['features'] = [
    feat for feat in states['features']
    if feat['id'] not in ['02', '72', '15']
]

### a. cow population map:

In [ ]:
custom_colors = ['rgb(237, 229, 207)',
                'rgb(224, 194, 162)',
                'rgb(211, 156, 131)',
                'rgb(193, 118, 111)',
                'rgb(166, 84, 97)',
                'rgb(129, 55, 83)',
                'rgb(84, 31, 63)','rgb(54, 19, 40)']

In [ ]:
for pop in ['usda','our']:
    if pop =='usda':
        ## reading USDA data:
        usda_cow = pd.read_csv('1_data/usda_nass_census_milk_cow_population_1997_2002_2007_2012_2017_2022_county_level_21May2025.csv', index_col=0)
        usda_cow = usda_cow[['Year','Geo Level','State','State ANSI','County ANSI','Data Item','Domain','Value']].copy().rename(columns={'Year':'year','Value':'cow','State ANSI':'state_id',
                                                                                                                                 'County ANSI':'GEOID'})
        print(usda_cow['Geo Level'].unique())
        usda_cow.loc[usda_cow['state_id'].isnull()].copy().reset_index(drop=True)
        usda_cow = usda_cow.loc[~usda_cow['GEOID'].isnull()].copy().reset_index(drop=True)
        usda_cow['state_id'] = usda_cow['state_id'].astype(int).astype(str).str.zfill(2)
        usda_cow['GEOID'] = usda_cow['state_id'] + usda_cow['GEOID'].astype(int).astype(str).str.zfill(3)
        print('remove D :', usda_cow.loc[usda_cow['cow'] != ' (D)'].shape[0])
        usda_cow = usda_cow.loc[usda_cow['cow'] != ' (D)'].copy().reset_index(drop=True)
        usda_cow['cow'] = usda_cow['cow'].replace(",","", regex=True).astype('float64')
        usda_cow = usda_cow.groupby(['State','state_id','GEOID'])['cow'].mean().reset_index().rename(columns={'State':'state_abv','cow':'id'})
        plot_df = usda_cow.copy()
        plot_df = plot_df.loc[plot_df['GEOID'].isin(df['GEOID'].unique())].copy()
    
    elif pop == 'our':
        plot_df = df.groupby(['state_abv','GEOID'])['id'].nunique().reset_index()

    bin_labels=['<100','100-500','500-1,000','1,000-2,500','2,500-5,000','5,000-10,000','10,000-210,000','>210,000']
    plot_df['bins'] = pd.cut(plot_df['id'], bins=[3,100,500,1000,2500,5000,10000,210000,440000], 
                             labels=bin_labels, retbins=True, right=False)[0]
    plot_df = plot_df.sort_values(by='bins',ascending=False)
    
    if np.nan in plot_df['bins'].unique():
        raise ExceptionType("Nan Value Exist!")
    
    color_map = dict(zip(bin_labels, custom_colors))    
    plot_df['nonee'] = np.nan
    
    # categorical
    fig = px.choropleth(plot_df, geojson=counties, locations='GEOID',
                        color=plot_df['bins'].astype(str),
                        scope='usa', labels={'color':'Cows (Heads)'},
                        color_discrete_map=color_map,
                       )
    fig.update_traces(marker_line_width=0.3)

    chor4 = go.Choropleth(locationmode='USA-states',locations=plot_df[['state_abv','nonee']].drop_duplicates()['state_abv'],
                        z=plot_df[['state_abv','nonee']].drop_duplicates()['nonee'],
                        colorscale = [[0,'rgba(0,0,0,0)'],[1,'rgba(0,0,0,0)']], showscale=False, 
                             marker_line_width=0.6,marker_line_color='black')   
    fig.add_trace(chor4)
    fig.update_geos(visible=False, scope='usa', 
                   showsubunits=True, subunitcolor='black', subunitwidth=1)

    fig.write_image(f'3_output/fig/fig_SI_{pop}_common_cows_county_level.png',scale=3)
    fig.show()

### b. milk yield map:
- USDA doesn't report energy corrected milk, so plot ours only

In [ ]:
plot_df = df.groupby(['state_abv','GEOID'])['adj_milk_kg'].mean().reset_index()
plot_df['nonee'] = np.nan

In [ ]:
fig = go.Figure(go.Choropleth( locationmode='geojson-id', geojson=counties,
                              locations=plot_df['GEOID'],
                              z=plot_df['adj_milk_kg'], 
                              zmin=15, zmax=50,
                              colorscale="Brwnyl",
                              colorbar={'outlinecolor':'black', 'outlinewidth':2,'tickfont':dict(size=20,family='Arial'),
                                       'orientation':'h','xanchor':'center','y':-0.45,'x':0.46,'len':0.6,'thickness':20},
                              marker_line_width=0.2,
                              colorbar_title=dict(text='milk (kg/day/head)', font_family='Arial', side='top',
                                                  font_size=15)
                             ))

for feature in states['features']:
    geom_type = feature['geometry']['type']
    coords = feature['geometry']['coordinates']

    if geom_type == 'Polygon':
        # Single polygon: list of rings
        rings = coords
    elif geom_type == 'MultiPolygon':
        # Multiple polygons: list of list of rings
        rings = [ring for polygon in coords for ring in polygon]
    else:
        continue  # Skip if not a polygon

    for ring in rings:
        try:
            lons, lats = zip(*ring)
            fig.add_trace(go.Scattergeo(
                lon=list(lons),
                lat=list(lats),
                mode='lines',
                line=dict(color='black', width=0.6),
                showlegend=False
            ))
        except Exception:
            continue  # In case of bad geometry

# 4. Finalize layout
fig.update_geos(
    visible=False,
    scope='usa',
    showsubunits=False,  # county-level done by choropleth
    resolution=110
)

fig.update_layout(margin=dict(l=0,r=0,t=0,b=0), paper_bgcolor="white")
fig.write_image('3_output/fig/fig_SI_milk_yield_county_level.png',scale=3)
fig.show()

# 3. Detrending herd-specific technological trends
## 3-1. Creating variables:

In [ ]:
## process_df
df = df.drop(columns=['dim_count'])
df['ln_milk'] = np.log(df['adj_milk_kg'])
df['year_norm'] = df['adj_year'].astype(int).astype(str).str[2:4].astype(int)
df['year_norm_2'] = df['year_norm'] **2

## 3-2. Fitting the herd-level models:
- For cross validation, the herd-specific technolgoical trend must be fit using only the training period for that fold. Therefore, multiple fits across training folds are done.
- After identifying the final best model, we use herd_milk_resid.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [ ]:

def log_message(message):
    """Writes log messages to a file and prints them (for SLURM output)"""
    print(message, flush=True) 

    

def fit_herd_trend(herd_id, group, fit_col, resid_col, train_end, flag_col,
                   cutoff=2.5, min_unique_years_lin=3):
    
    key_cols = ['geoid_herd','id','calvingdate','date']
    group[fit_col] = np.nan
    group[resid_col] = np.nan
    group[flag_col] = np.nan

    train_mask = group["adj_year"] <= train_end
    val_mask = group["adj_year"] > train_end

    # --- ensure there are rows to fit quadratic ---
    fit_q = train_mask & group[["year_norm", "year_norm_2", "ln_milk"]].notna().all(axis=1)
    fit_l = train_mask & group[["year_norm", "ln_milk"]].notna().all(axis=1)

    n_years = group.loc[fit_l, "year_norm"].nunique()

    # If nothing to fit on, go to mean fallback immediately
    if fit_l.sum() == 0:
        group[flag_col] = "3_avg"
        herd_mean_train = group.loc[train_mask & group["ln_milk"].notna()].groupby("adj_year")["ln_milk"].mean().mean()
        group.loc[train_mask, fit_col] = herd_mean_train
        if train_end != 2024:
            herd_mean_val = group.loc[val_mask & group["ln_milk"].notna()].groupby("adj_year")["ln_milk"].mean().mean()
            group.loc[val_mask, fit_col] = herd_mean_val
        group[resid_col] = group["ln_milk"] - group[fit_col]
        return group

    # --- 1) Quadratic fit (only if enough years and enough rows) ---
    if (n_years >= min_unique_years_lin) and (fit_q.sum() > 0):
        herd_model = LinearRegression().fit(
            group.loc[fit_q, ["year_norm", "year_norm_2"]],
            group.loc[fit_q, "ln_milk"]
        )

        # predict train rows that have features
        pred_q_train = train_mask & group[["year_norm", "year_norm_2"]].notna().all(axis=1)
        if pred_q_train.any():
            ptrain = herd_model.predict(group.loc[pred_q_train, ["year_norm", "year_norm_2"]])
            group.loc[pred_q_train, fit_col] = ptrain

        # predict val rows that have features
        if train_end != 2024:
            pred_q_val = val_mask & group[["year_norm", "year_norm_2"]].notna().all(axis=1)
            if pred_q_val.any():
                pval = herd_model.predict(group.loc[pred_q_val, ["year_norm", "year_norm_2"]])
                group.loc[pred_q_val, fit_col] = pval

        group[flag_col] = "1_quad_fit"
        group[resid_col] = group["ln_milk"] - group[fit_col]

        extreme = (train_mask & group[resid_col].notna() & (group[resid_col].abs() > cutoff))
        if not extreme.any():
            return group

        print(f"Extremely large residuals for herd {group['geoid_herd'].iloc[0]}")

    # --- 2) Linear fallback ---
    if n_years >= min_unique_years_lin:
        herd_model2 = LinearRegression().fit(
            group.loc[fit_l, ["year_norm"]],
            group.loc[fit_l, "ln_milk"]
        )

        pred_l_train = train_mask & group[["year_norm"]].notna().all(axis=1)
        if pred_l_train.any():
            ptrain = herd_model2.predict(group.loc[pred_l_train, ["year_norm"]])
            group.loc[pred_l_train, fit_col] = ptrain

        if train_end != 2024:
            pred_l_val = val_mask & group[["year_norm"]].notna().all(axis=1)
            if pred_l_val.any():
                pval = herd_model2.predict(group.loc[pred_l_val, ["year_norm"]])
                group.loc[pred_l_val, fit_col] = pval

        group[flag_col] = "2_lin_fit"
        group[resid_col] = group["ln_milk"] - group[fit_col]

        extreme2 = (train_mask & group[resid_col].notna() & (group[resid_col].abs() > cutoff))
        if not extreme2.any():
            return group.set_index(key_cols)[[fit_col, resid_col, flag_col]]

    # --- 3) Mean fallback ---
    group[flag_col] = "3_avg"
    herd_mean_train = group.loc[train_mask & group["ln_milk"].notna()].groupby("adj_year")["ln_milk"].mean().mean()
    group.loc[train_mask, fit_col] = herd_mean_train
    if train_end != 2024:
        herd_mean_val = group.loc[val_mask & group["ln_milk"].notna()].groupby("adj_year")["ln_milk"].mean().mean()
        group.loc[val_mask, fit_col] = herd_mean_val
    group[resid_col] = group["ln_milk"] - group[fit_col]
    return group


In [ ]:
## fitting:
key_cols = ["adj_year", "geoid_herd", "id", "calvingdate", "date"]
end_years = [2009, 2012, 2015, 2018, 2021, 2024, 2027]

def split_names(end_year, fold_idx):
    train_end = end_year - 3

    if end_year == 2024:
        suffix = "_train"
    elif end_year == 2027:
        suffix = ""   # final
    else:
        suffix = "_" + str(fold_idx)

    fit_col  = f"fitted_herd{suffix}"
    resid_col = f"herd_milk_resid{suffix}"
    flag_col = f"herd_trend_flag{suffix}"
    return train_end, fit_col, resid_col, flag_col

out = []
for fold_idx, end_year in enumerate(end_years, start=1):
    print(end_year)
    train_end, fit_col, resid_col, flag_col = split_names(end_year, fold_idx)

    print(f"train_end_yr: {train_end} | val_end_yr: {end_year}")
    print("cols:", fit_col, resid_col, flag_col)

    df_slice = df.loc[df["adj_year"] <= end_year][key_cols + ['ln_milk','year_norm','year_norm_2']].copy()

    for herd_id, group in df_slice.groupby("geoid_herd", sort=False):
        out.append(
            fit_herd_trend(
                herd_id, group, fit_col, resid_col, train_end,
                flag_col=flag_col, cutoff=2.5, min_unique_years_lin=3
            )
        )

    # IMPORTANT: keep original index
    df_results = pd.concat(out)

    # Attach by index (no merge)
    df.loc[df_results.index, [fit_col, resid_col, flag_col]] = df_results[[fit_col, resid_col, flag_col]]

In [ ]:
## sanity check:
for herd in df['geoid_herd'].unique():
    fig, ax = plt.subplots(figsize=(3,2), tight_layout=True)
    df.loc[df['geoid_herd'] == herd].groupby(['adj_year'])['ln_milk'].mean().plot(marker='o', linestyle='', ax=ax, markersize=2)
    df.loc[df['geoid_herd'] == herd].groupby(['adj_year'])['fitted_herd'].mean().plot(ax=ax)
    plt.show()

In [ ]:
del out
gc.collect()

In [ ]:
## saving:
df.to_parquet('3_output/2_final_herd_detrend_df.gzip', compression='gzip')